# Notes

In [1]:
import sys
import os
import pandas as pd

import numpy as np
from scipy import stats

import taxonomia
import librosa.display

from data_engineering import process_features
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [2]:
# Obtener la ruta del directorio raíz
root_dir = os.path.abspath(os.path.join(os.getcwd(), '..')) 

In [3]:
familias = os.listdir('songs/')
familias.sort()
for familia in familias:
    generos_path = f'songs/{familia}/'
    generos = os.listdir(generos_path)
    generos.sort()
    
    for genero in generos:
        especies_path = f'{generos_path}{genero}/'
        if os.path.isdir(especies_path):
            especies = [e for e in os.listdir(especies_path) if os.path.isdir(os.path.join(especies_path, e))]
            
            for especie in especies:
                name_comun_path = f'{especies_path}{especie}/'
                name_comuns = [nc for nc in os.listdir(name_comun_path) if os.path.isdir(os.path.join(name_comun_path, nc))]
                name_comuns.sort()
                
                for name_comun in name_comuns:
                    ruta_base = f'{name_comun_path}{name_comun}'
                    nombre_archivo = f"./data/{familia}_{genero}_{especie}.csv"
                    if not os.path.exists(nombre_archivo):
                        print(f'Procesando: {nombre_archivo}')
                        print(f'\t\t\tRuta base: {ruta_base}')
                        process_features(
                        f"{ruta_base}/",
                        "./data/"
                        f"{familia}_{genero}_{especie}"
                        )
                    else:
                        #pass
                        print(f'El archivo {nombre_archivo} ya existe, saltando procesamiento.')

El archivo ./data/Alaudidae_Eremophila_Eremophila alpestris.csv ya existe, saltando procesamiento.
El archivo ./data/Bombycillidae_Bombycilla_Bombycilla cedrorum.csv ya existe, saltando procesamiento.
El archivo ./data/Capitonidae_Capito_Capito auratus.csv ya existe, saltando procesamiento.
El archivo ./data/Capitonidae_Eubucco_Eubucco richardsoni.csv ya existe, saltando procesamiento.
El archivo ./data/Cinclidae_Cinclus_Cinclus leucocephalus.csv ya existe, saltando procesamiento.
El archivo ./data/Conopophagidae_Conopophaga_Conopophaga castaneiceps.csv ya existe, saltando procesamiento.
El archivo ./data/Conopophagidae_Conopophaga_Conopophaga aurita.csv ya existe, saltando procesamiento.
El archivo ./data/Conopophagidae_Conopophaga_Conopophaga peruviana.csv ya existe, saltando procesamiento.
El archivo ./data/Conopophagidae_Pittasoma_Pittasoma rufopileatum.csv ya existe, saltando procesamiento.
El archivo ./data/Conopophagidae_Pittasoma_Pittasoma michleri.csv ya existe, saltando proce

In [4]:
# process_features(
#     "songs/Alaudidae/Eremophila/Eremophila alpestris/HornedLark/",
#     "./data/"
#     "ElegantWoodcreeper"
# )

In [4]:
import os
import pandas as pd

ruta_features = './data/'
archivos_csv = os.listdir(ruta_features)
dfs = []

for archivo_csv in archivos_csv:
    # Extraer familia, género y especie del nombre del archivo
    partes_nombre = archivo_csv.split('_')
    familia = partes_nombre[0]
    genero = partes_nombre[1]
    especie = '_'.join(partes_nombre[2:]).replace('.csv', '')  # Asume que el nombre de la especie puede tener guiones bajos

    try:
        df = pd.read_csv(os.path.join(ruta_features, archivo_csv), encoding='latin1')
        # Agregar las columnas con la información extraída
        df['Family'] = familia
        df['Genus'] = genero
        df['Specie'] = especie

        dfs.append(df)
    except UnicodeDecodeError as e:
        print(f"Error al leer el archivo {archivo_csv}: {e}")

df_final = pd.concat(dfs, ignore_index=True)
print(df_final.shape)

(89875, 4821)


In [5]:
df_final['Suborder'] = df_final['Family'].map(taxonomia.suborder)
df_final = df_final.drop(columns=['Unnamed: 0'])

In [6]:
df_final = df_final[df_final['Family'] != '.DS']

In [10]:
df_final['Family'].isnull().mean()

0.0

In [18]:
df_final['Suborder'].isnull().mean()

0.01733615221987315

In [16]:
df_final[df_final['Suborder'].isnull()]['Family'].unique()

array(['Psittacidae', 'Falconidae', 'Capitonidae', 'Rhamphastidae',
       'Picidae'], dtype=object)

In [19]:
df_final['Genus'].isnull().mean()

0.0

In [20]:
df_final['Specie'].isnull().mean()

0.0

In [8]:
df_final.isnull().mean().sort_values(ascending=False).head(10)

tempogram_skew_01              0.998587
tempogram_kurtosis_01          0.998587
mel_spectrogram_kurtosis_01    0.022355
mel_spectrogram_kurtosis_02    0.022299
mel_spectrogram_kurtosis_03    0.022288
mel_spectrogram_kurtosis_04    0.022265
mel_spectrogram_kurtosis_10    0.022221
mel_spectrogram_kurtosis_95    0.022210
mel_spectrogram_kurtosis_05    0.022210
mel_spectrogram_kurtosis_99    0.022210
dtype: float64

In [21]:
df_final.to_csv('/data/species_features.csv', index=False)

In [24]:
pd.read_csv('/data/species_features.csv').shape

(89870, 4821)